# 实验二 · 第二阶段：房源采集与 CSV 数据存储

采集安居客教学站 `http://www.bspider.top/anjuke/` 的房源信息，解析 8 个字段，
翻页采集全部房源，保存为 `data/houseinfo.csv`。

本 Notebook 分 4 步：

1. 请求网页并验证返回内容
2. 分析网页结构，定位 8 个字段
3. 翻页采集
4. 保存并核对 CSV

## 0. 安装依赖

BeautifulSoup 的安装名是 `beautifulsoup4`，导入名是 `bs4`。

In [1]:
%pip install requests beautifulsoup4 pymysql

Defaulting to user installation because normal site-packages is not writeable


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


Note: you may need to restart the kernel to use updated packages.


## 1. 请求第一页并检查返回内容

### 1.1 反爬注意事项（本次实验踩到的坑）

教学站有反爬校验。如果 `User-Agent` 写成简写的 `"Mozilla/5.0"`，
服务器会返回**访问验证页**（HTML 里含 `Antispam`），HTTP 状态码依然是 `200`，
但里面一条房源都没有。

所以必须伪装成完整浏览器 UA，并且**每次请求后都检查返回内容**，不能只看状态码。

- 反爬验证页：HTML 中出现 `Antispam`
- 正常列表页：能匹配到 `div.property`，每页 60 条

In [2]:
import csv
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"

# 必须是完整浏览器 UA，简写 "Mozilla/5.0" 会被反爬拦截，返回访问验证页
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
}

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

In [3]:
# 请求第一页
r = requests.get(BASE_URL, headers=HEADERS, timeout=60)
r.raise_for_status()                   # HTTP 错误时抛异常
r.encoding = r.apparent_encoding       # 保证中文不乱码

print("状态码：", r.status_code)
print("网页标题：", BeautifulSoup(r.text, "html.parser").title.get_text(strip=True))
print("是否访问验证页：", "Antispam" in r.text)
print("HTML 长度：", len(r.text))

soup = BeautifulSoup(r.text, "html.parser")

状态码： 200
网页标题： 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
是否访问验证页： False
HTML 长度： 280794


### 1.2 自己检查

- 状态码是 200 **不等于**内容正确，必须确认能匹配到房源节点。
- 下面这一行必须输出 60，如果输出 0，说明拿到的是验证页或选择器不对。

In [4]:
rows = soup.select("div.property")
print("第 1 页房源条数：", len(rows))
assert len(rows) > 0, "没有匹配到房源，可能被反爬拦截或选择器有误"

第 1 页房源条数： 60


## 2. 分析网页结构，定位 8 个字段

### 2.1 找到"一条房源"的父容器

在 Chrome 中按 F12 → Elements → 选取元素按钮 → 点击第一条房源标题，
可以看到整条房源被一个 `div.property` 包住。它同时包含标题、楼盘、地址、
属性组和价格，正好是一条房源的完整范围。第二条房源用的是同样的 class。

用 `select()` 取出所有匹配节点（返回列表），用 `select_one()` 取第一个匹配节点
（找不到返回 `None`）。

**相对选择器的思路**：先选中一条房源，再在它内部找字段，
不必保留从页面根节点开始的长路径（长路径里的 `nth-child()` 一旦页面多插入一个元素就失效）。

In [5]:
# 打印第一条房源的顶层结构，确认 div.property 刚好包住整条房源
row = rows[0]
print("第一条房源包含的 class：")
for child in row.find_all(True, recursive=False):
    print("   ", child.name, child.get("class"))

第一条房源包含的 class：
    a ['property-ex']


### 2.2 逐个字段定位

先打印一条房源里各字段的原始文字，再决定怎么取。

In [6]:
# 标题
print("h3 标题：", repr(row.select_one("h3").get_text(strip=True)))

# 楼盘
print("楼盘：", repr(row.select_one("p.property-content-info-comm-name").get_text(strip=True)))

# 地址：文字分散在 3 个 span 里，拼起来才是完整地址
addr_node = row.select_one("p.property-content-info-comm-address")
print("地址 span 列表：", [s.get_text(strip=True) for s in addr_node.select("span")])
print("地址拼接后：", repr("".join(s.get_text(strip=True) for s in addr_node.select("span"))))

# 属性组：5 个 p，顺序固定
info = row.select_one("div.property-content-info:not(.property-content-info-comm)")
items = info.select("p")
for i, it in enumerate(items):
    print(f"  属性组 p[{i}] class={it.get('class')} text={it.get_text(strip=True)!r}")

# 单价 vs 总价：两个不同的节点
print("单价（每平方米）：", repr(row.select_one("p.property-price-average").get_text(strip=True)))
print("总价（注意不是本实验要的）：", repr(row.select_one("p.property-price-total").get_text(strip=True)))

h3 标题： '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
楼盘： '万科金域学府翰林菁华里'
地址 span 列表： ['南岸', '学府大道', '二塘路59号']
地址拼接后： '南岸学府大道二塘路59号'
  属性组 p[0] class=['property-content-info-text', 'property-content-info-attribute'] text='3室2厅1卫'
  属性组 p[1] class=['property-content-info-text'] text='89.9㎡'
  属性组 p[2] class=['property-content-info-text'] text='南北'
  属性组 p[3] class=['property-content-info-text'] text='低层(共27层)'
  属性组 p[4] class=['property-content-info-text'] text='2019年建造'
单价（每平方米）： '16465元/㎡'
总价（注意不是本实验要的）： '148万'


### 2.3 三个容易取错的字段

**属性组 5 个 p 的顺序（本次实测 600 条全部一致）**：

| 下标 | 内容 | 示例 | 取法 |
|---|---|---|---|
| p[0] | 户型 | `3室2厅1卫` | 用 `p.property-content-info-attribute` 单独取更稳 |
| p[1] | 面积 | `89.9㎡` | 含 `㎡` 的那项 |
| p[2] | 朝向 | `南北` | **不是楼层**，不能用 |
| p[3] | 楼层 | `低层(共27层)` | 下标 3 |
| p[4] | 建造年代 | `2019年建造` | 含"年建造"的那项 |

- **面积**：按含 `㎡`/`平米` 特征找，不写死下标。
- **楼层**：p[2] 是朝向（南北/东西），p[3] 才是楼层（低层/中层/高层 + 共 N 层）。
- **建造年代**：节点可能缺失，找不到时填空字符串，**不要直接跳过这条记录**。
- **单价**：必须取 `p.property-price-average`（`16465元/㎡`），
  不是 `p.property-price-total`（`148万`，那是总价）。

In [7]:
def parse_row(row):
    """把一条房源容器解析成 dict，字段顺序与 FIELDS 一致。"""

    # 属性区。页面上有两个 div.property-content-info，
    # :not(.property-content-info-comm) 排除掉装楼盘/地址的那个
    info = row.select_one("div.property-content-info:not(.property-content-info-comm)")
    items = info.select("p") if info else []

    def item_text(index):
        # 用下标取值前先检查长度，防止越界
        if 0 <= index < len(items):
            return items[index].get_text(strip=True)
        return ""

    # 户型：单独按 class 取
    struct_node = row.select_one("p.property-content-info-attribute")
    struct = struct_node.get_text(strip=True) if struct_node else ""

    # 面积：按单位特征找
    area = ""
    for item in items:
        text = item.get_text(strip=True)
        if "㎡" in text or "平米" in text:
            area = text
            break

    # 楼层：p[3]（p[2] 是朝向，不能取）
    floor = item_text(3)

    # 建造年代：按特征找，缺失留空
    maketime = ""
    for item in items:
        text = item.get_text(strip=True)
        if "年建造" in text:
            maketime = text
            break

    # 单价：property-price-average 才是每平方米单价
    price_node = row.select_one("p.property-price-average")
    price = price_node.get_text(strip=True) if price_node else ""

    # 地址：3 个 span 拼起来，中间不额外插空格
    address_node = row.select_one("p.property-content-info-comm-address")
    address = ""
    if address_node:
        address = "".join(s.get_text(strip=True) for s in address_node.select("span"))

    house_node = row.select_one("p.property-content-info-comm-name")
    house = house_node.get_text(strip=True) if house_node else ""

    title_node = row.select_one("h3")
    title = title_node.get_text(strip=True) if title_node else ""

    return {
        "title": title,
        "house": house,
        "address": address,
        "struct": struct,
        "area": area,
        "floor": floor,
        "maketime": maketime,
        "price": price,
    }

### 2.4 先解析一条，再逐字段对照网页

按 `title → house → address → struct → area → floor → maketime → price` 的顺序输出，
和网页上的显示内容逐个核对。然后检查第 2 条和最后 1 条。

In [8]:
for idx, label in [(0, "第 1 条"), (1, "第 2 条"), (len(rows) - 1, "最后 1 条")]:
    rec = parse_row(rows[idx])
    print(f"--- {label} ---")
    for f in FIELDS:
        print(f"  {f:9s} = {rec[f]!r}")

--- 第 1 条 ---
  title     = '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  house     = '万科金域学府翰林菁华里'
  address   = '南岸学府大道二塘路59号'
  struct    = '3室2厅1卫'
  area      = '89.9㎡'
  floor     = '低层(共27层)'
  maketime  = '2019年建造'
  price     = '16465元/㎡'
--- 第 2 条 ---
  title     = '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住'
  house     = '两代一家'
  address   = '渝北渝北黄泥磅紫荆路29号'
  struct    = '3室2厅2卫'
  area      = '94.5㎡'
  floor     = '低层(共28层)'
  maketime  = '2008年建造'
  price     = '14815元/㎡'
--- 最后 1 条 ---
  title     = '精装修 电梯房 4室2厅 91平'
  house     = '华润凯旋天地'
  address   = '南岸弹子石腾龙大道45号'
  struct    = '4室2厅2卫'
  area      = '91.3㎡'
  floor     = '低层(共32层)'
  maketime  = '2015年建造'
  price     = '15886元/㎡'


## 3. 翻页采集

### 3.1 找"下一页"链接

在网页底部右键"下一页" → 检查，可以看到它的 class 是 `next next-active`，
`href` 是 `?page=2`——只有查询参数，必须用 `urljoin` 配合当前页地址补全成绝对地址。

注意：`get_text()` 取得的是"下一页"这几个字，`get("href")` 取得的才是地址。

In [9]:
next_link = soup.select_one("a.next.next-active")
print("下一页 href 原值：", repr(next_link.get("href")))
print("get_text() 得到的是：", repr(next_link.get_text(strip=True)))

next_url = urljoin(BASE_URL, next_link.get("href"))
print("urljoin 补全后：", next_url)

下一页 href 原值： '?page=2'
get_text() 得到的是： '下一页'
urljoin 补全后： http://www.bspider.top/anjuke/?page=2


### 3.2 翻页循环

每轮做 5 件事：

1. 请求当前页，检查响应内容
2. 解析本页房源，加入结果列表
3. 找下一页并补全地址
4. 检查地址没有越出教学站、也没有访问过
5. 等待至少 1 秒，再请求下一页

**停止条件**：末页的"下一页"按钮 `next-active` class 会被去掉
（只剩 `class="next"`、`href="#"`），此时 `select_one("a.next.next-active")`
返回 `None`。整个分页区域消失或找不到按钮属于结构异常，不能算采集完成。

**结果列表放在函数内部**（`records = []`），不放在全局：
函数内的局部变量不会污染模块命名空间，重复调用不会互相污染，
也便于测试和传参。

In [10]:
def fetch(url, retries=5):
    """请求页面并返回 soup，失败自动重试。"""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            r.raise_for_status()
            r.encoding = r.apparent_encoding
            if "Antispam" in r.text:
                raise RuntimeError("返回的是访问验证页，不是房源列表")
            return BeautifulSoup(r.text, "html.parser")
        except Exception as exc:
            last_error = exc
            print(f"    请求失败（第 {attempt}/{retries} 次）：{exc}")
            time.sleep(3)
    raise RuntimeError(f"无法获取页面：{url}，最后一次错误：{last_error}")

In [11]:
MAX_PAGES = 30          # 安全页数上限，防止无限循环
REQUEST_INTERVAL = 1.2  # 每次翻页至少等待 1 秒


def next_page_url(soup, current_url):
    """返回下一页地址；已到末页时返回 None。"""
    link = soup.select_one("a.next.next-active")
    if link is None:
        return None
    href = link.get("href")
    if not href:
        return None
    return urljoin(current_url, href)


def collect():
    """逐页采集，返回 (记录列表, 页数)。"""
    records = []            # 结果列表放在函数内部，不用全局变量
    visited = set()
    current_url = BASE_URL
    page_no = 0

    while True:
        page_no += 1
        if page_no > MAX_PAGES:
            raise RuntimeError(f"达到安全页数上限 {MAX_PAGES}，采集中止")

        if current_url in visited:
            raise RuntimeError(f"地址重复，疑似陷入循环：{current_url}")
        visited.add(current_url)

        print(f"[第 {page_no} 页] 请求 {current_url}")
        soup = fetch(current_url)

        rows = soup.select("div.property")
        if not rows:
            raise RuntimeError(f"第 {page_no} 页未解析到房源，采集中止")

        for row in rows:
            records.append(parse_row(row))

        print(f"    本页 {len(rows)} 条，累计 {len(records)} 条")

        nxt = next_page_url(soup, current_url)
        if nxt is None:
            print("    下一页按钮已禁用或消失 -> 到达末页")
            break
        if nxt in visited:
            raise RuntimeError(f"下一页地址已访问过，疑似重复：{nxt}")
        if not nxt.startswith("http://www.bspider.top/anjuke/"):
            raise RuntimeError(f"地址越出教学站：{nxt}")

        print(f"    下一页：{nxt}")
        time.sleep(REQUEST_INTERVAL)
        current_url = nxt

    return records, page_no

### 3.3 先试采 2 页，确认第二页没有重复第一页

In [12]:
# 先只采 2 页做小规模验证
def collect_limited(max_pages):
    records = []
    visited = set()
    current_url = BASE_URL
    for page_no in range(1, max_pages + 1):
        if current_url in visited:
            raise RuntimeError(f"地址重复：{current_url}")
        visited.add(current_url)
        print(f"[第 {page_no} 页] {current_url}")
        soup = fetch(current_url)
        rows = soup.select("div.property")
        for row in rows:
            records.append(parse_row(row))
        print(f"    本页 {len(rows)} 条，累计 {len(records)} 条")
        nxt = next_page_url(soup, current_url)
        if nxt is None:
            break
        time.sleep(REQUEST_INTERVAL)
        current_url = nxt
    return records


test_records = collect_limited(2)
print()
print("两页合计：", len(test_records), "条")
p1 = [tuple(r[f] for f in FIELDS) for r in test_records[:60]]
p2 = [tuple(r[f] for f in FIELDS) for r in test_records[60:]]
print("第 1 页与第 2 页是否有完全相同的记录：", len(set(p1) & set(p2)))

[第 1 页] http://www.bspider.top/anjuke/


    本页 60 条，累计 60 条


[第 2 页] http://www.bspider.top/anjuke/?page=2


    本页 60 条，累计 120 条



两页合计： 120 条
第 1 页与第 2 页是否有完全相同的记录： 0


### 3.4 完整采集

In [13]:
records, pages = collect()
print()
print(f"成功采集页数：{pages}")
print(f"采集记录数（去重前）：{len(records)}")

[第 1 页] 请求 http://www.bspider.top/anjuke/


    本页 60 条，累计 60 条
    下一页：http://www.bspider.top/anjuke/?page=2


[第 2 页] 请求 http://www.bspider.top/anjuke/?page=2


    本页 60 条，累计 120 条
    下一页：http://www.bspider.top/anjuke/?page=3


[第 3 页] 请求 http://www.bspider.top/anjuke/?page=3


    本页 60 条，累计 180 条
    下一页：http://www.bspider.top/anjuke/?page=4


... (10 outputs left out)

[第 9 页] 请求 http://www.bspider.top/anjuke/?page=9


    本页 60 条，累计 540 条
    下一页：http://www.bspider.top/anjuke/?page=10


[第 10 页] 请求 http://www.bspider.top/anjuke/?page=10


    本页 60 条，累计 600 条
    下一页按钮已禁用或消失 -> 到达末页

成功采集页数：10
采集记录数（去重前）：600


## 4. 保存并核对 CSV

要点：

- 写入工具用 `csv.writer`
- 编码 `utf-8-sig`（带 BOM，Excel 打开不乱码），`newline=""`
- 表头 8 个字段名，顺序一致
- 每条记录恰好 8 项
- 按 8 个字段完全相同去重，并记录去重前后条数

In [14]:
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "houseinfo.csv"

# 按 8 个字段完全相同去重
seen = set()
unique = []
for rec in records:
    key = tuple(rec[f] for f in FIELDS)
    if key in seen:
        continue
    seen.add(key)
    unique.append(rec)

dup_count = len(records) - len(unique)
print(f"去重前 {len(records)} 条，去掉完全相同记录 {dup_count} 条，剩余 {len(unique)} 条")

DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for rec in unique:
        writer.writerow([rec[f] for f in FIELDS])

print("已保存：", CSV_PATH.resolve())

去重前 600 条，去掉完全相同记录 102 条，剩余 498 条
已保存： /home/pwuser/workspace/L02_reliable_collection/data/houseinfo.csv


### 4.1 回读核对

用 `csv.DictReader` 重新读取文件。**表头不计入数据行数**。

In [15]:
with open(CSV_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    rows_read = list(reader)
    print("回读表头：", reader.fieldnames)

print("回读数据行数：", len(rows_read))

# 选第一条、跨页一条和最后一条，逐字段打印
for idx, label in [(0, "第 1 条"), (len(rows_read) // 2, "跨页一条"), (len(rows_read) - 1, "最后 1 条")]:
    print(f"\n--- {label}（行 {idx}）---")
    for f in FIELDS:
        print(f"  {f:9s} = {rows_read[idx][f]!r}")

回读表头： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
回读数据行数： 498

--- 第 1 条（行 0）---
  title     = '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  house     = '万科金域学府翰林菁华里'
  address   = '南岸学府大道二塘路59号'
  struct    = '3室2厅1卫'
  area      = '89.9㎡'
  floor     = '低层(共27层)'
  maketime  = '2019年建造'
  price     = '16465元/㎡'

--- 跨页一条（行 249）---
  title     = '精装修洋房 位置安静 全新装修 随时看房'
  house     = '协信城立方玺院'
  address   = '沙坪坝大学城大学城南二路'
  struct    = '3室3厅2卫'
  area      = '95.8㎡'
  floor     = '中层(共8层)'
  maketime  = '2016年建造'
  price     = '15658元/㎡'

--- 最后 1 条（行 497）---
  title     = '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售'
  house     = '恒大照母山山水城'
  address   = '渝北照母山恒康路9号'
  struct    = '3室2厅1卫'
  area      = '90.2㎡'
  floor     = '低层(共32层)'
  maketime  = '2015年建造'
  price     = '15530元/㎡'


### 4.2 一致性检查

确认：每行都是 8 个字段、地址完整、面积带单位、楼层不是朝向、单价不是总价。

In [16]:
import re

bad = {f: 0 for f in FIELDS}
for r in rows_read:
    if not r["title"]: bad["title"] += 1
    if not r["house"]: bad["house"] += 1
    if not r["address"]: bad["address"] += 1
    if not re.match(r"^\d+室\d+厅\d+卫$", r["struct"]): bad["struct"] += 1
    if not re.search(r"[㎡平米]", r["area"]): bad["area"] += 1
    if not re.search(r"层", r["floor"]): bad["floor"] += 1
    if not re.search(r"\d{4}年建造", r["maketime"]): bad["maketime"] += 1
    if not re.search(r"\d+元/㎡", r["price"]): bad["price"] += 1

print("不符合预期的行数：", bad)
print("楼层误取成朝向的行数：", sum(1 for r in rows_read if r["floor"] in ("南北", "东西")))
print("单价误取成总价的行数：", sum(1 for r in rows_read if "万" in r["price"]))

keys = [tuple(r[f] for f in FIELDS) for r in rows_read]
print("CSV 中残留的完全相同记录数：", len(keys) - len(set(keys)))

不符合预期的行数： {'title': 0, 'house': 0, 'address': 0, 'struct': 0, 'area': 0, 'floor': 0, 'maketime': 0, 'price': 0}
楼层误取成朝向的行数： 0
单价误取成总价的行数： 0
CSV 中残留的完全相同记录数： 0


### 4.3 汇总

In [17]:
print("===== 第二阶段汇总 =====")
print(f"成功采集页数：{pages}")
print(f"采集记录数（去重前）：{len(records)}")
print(f"完全相同的重复记录数：{dup_count}")
print(f"CSV 数据行数（不含表头）：{len(rows_read)}")
print(f"CSV 路径：{CSV_PATH.resolve()}")

===== 第二阶段汇总 =====
成功采集页数：10
采集记录数（去重前）：600
完全相同的重复记录数：102
CSV 数据行数（不含表头）：498
CSV 路径：/home/pwuser/workspace/L02_reliable_collection/data/houseinfo.csv
